In [2]:
# Install required packages
!pip install --user --no-cache-dir catboost
!pip install --user xgboost lightgbm optuna scikit-learn pandas numpy

After installing the packages, please restart the kernel and run all cells from the beginning.

# Materials Property Prediction and Optimization

This notebook implements a machine learning pipeline to:
1. Train models to predict aluminum alloy properties (Elongation, Tensile Strength, Yield Strength)
2. Use the trained models to predict properties for new compositions
3. Compare and identify improved compositions

In [5]:
# Import required libraries
import pandas as pd
import numpy as np
import optuna
from sklearn.metrics import r2_score, mean_absolute_error, make_scorer
from sklearn.model_selection import cross_val_score, KFold
from sklearn.multioutput import MultiOutputRegressor
import xgboost as xgb
import lightgbm as lgb
import warnings
warnings.filterwarnings('ignore')

C:\Users\Akkaldevi SaiVinayak\AppData\Local\Packages\PythonSoftwareFoundation.Python.3.11_qbz5n2kfra8p0\LocalCache\local-packages\Python311\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [7]:
# Load the datasets
final_data = pd.read_csv('final_aluminum_data.csv')
new_compositions = pd.read_csv('new_compositions.csv')

# Display basic information about the datasets
print("Final Aluminum Data Shape:", final_data.shape)

print("\nNew Compositions Shape:", new_compositions.shape)

# Display the first few rows of each dataset
print("\nFinal Aluminum Data (first few rows):")
print(final_data.head())
print("\nNew Compositions (first few rows):")
print(new_compositions.head())

Final Aluminum Data Shape: (1154, 35)

New Compositions Shape: (100, 21)

Final Aluminum Data (first few rows):
   Unnamed: 0   Ag       Al    B   Be   Bi   Cd   Co   Cr      Cu  ...  \
0           0  0.0  0.88011  0.0  0.0  0.0  0.0  0.0  0.0  0.0198  ...   
1           1  0.0  0.88011  0.0  0.0  0.0  0.0  0.0  0.0  0.0198  ...   
2           2  0.0  0.99450  0.0  0.0  0.0  0.0  0.0  0.0  0.0000  ...   
3           3  0.0  0.99250  0.0  0.0  0.0  0.0  0.0  0.0  0.0000  ...   
4           4  0.0  0.99000  0.0  0.0  0.0  0.0  0.0  0.0  0.0000  ...   

       Zr  Elongation (%)  Tensile Strength (MPa)  Yield Strength (MPa)  \
0  0.0012            16.8                   651.6                 583.3   
1  0.0012            15.4                   557.0                 513.0   
2  0.0000            10.5                   320.0                 300.0   
3  0.0000             4.5                   280.0                 265.0   
4  0.0000             7.0                   325.0                 29

In [8]:
# Prepare features and target variables
# Define target variables
target_variables = ['Elongation (%)', 'Tensile Strength (MPa)', 'Yield Strength (MPa)']

# Get common columns between the two datasets
common_columns = set(final_data.columns) & set(new_compositions.columns)

# Remove target variables and unnecessary columns from features
feature_columns = [col for col in common_columns 
                  if col not in target_variables + ['Unnamed: 0', 'class']]

# Clean data: Remove rows with NaN or infinite values in target variables
mask = np.ones(len(final_data), dtype=bool)
for target in target_variables:
    mask = mask & ~final_data[target].isna() & ~np.isinf(final_data[target])
final_data_clean = final_data[mask].copy()

print(f"Removed {len(final_data) - len(final_data_clean)} rows with NaN or infinite values")

# Prepare the training data
X = final_data_clean[feature_columns]
y_dict = {target: final_data_clean[target] for target in target_variables}

# Prepare the new composition data
X_new = new_compositions[feature_columns]

# Clean features: Check for NaN values in features
if X.isna().any().any():
    print("\nWarning: NaN values found in features. Filling with 0.")
    X = X.fillna(0)
if X_new.isna().any().any():
    print("\nWarning: NaN values found in new compositions. Filling with 0.")
    X_new = X_new.fillna(0)  # Fill with 0 for consistency

print("\nFeature columns:", feature_columns)
print("\nNumber of features:", len(feature_columns))
print("Number of samples for training:", X.shape[0])
print("Number of new compositions to predict:", X_new.shape[0])

print("\nTarget variable statistics:")
for target in target_variables:
    print(f"\n{target}:")
    print(y_dict[target].describe())

# Display the first few rows of each dataset's features
print("\nFirst few rows of training features:")
print(X.head())
print("\nFirst few rows of new composition features:")
print(X_new.head())

Removed 219 rows with NaN or infinite values


Feature columns: ['Pb', 'Ni', 'Si', 'Sn', 'V', 'Mn', 'Cr', 'aging_time', 'solution_temp', 'Zr', 'Cu', 'strain_hardening_index', 'Mg', 'Ti', 'aging_temp', 'Al', 'Fe', 'solution_time', 'Zn', 'Co']

Number of features: 20
Number of samples for training: 935
Number of new compositions to predict: 100

Target variable statistics:

Elongation (%):
count    935.000000
mean      12.094428
std        7.171187
min        0.500000
25%        7.000000
50%       11.000000
75%       15.300000
max       50.000000
Name: Elongation (%), dtype: float64

Tensile Strength (MPa):
count    935.000000
mean     349.720207
std      152.021431
min       44.815920
25%      225.250000
50%      330.948336
75%      472.000000
max      820.000000
Name: Tensile Strength (MPa), dtype: float64

Yield Strength (MPa):
count    935.000000
mean     281.453196
std      151.105503
min       10.342135
25%      159.289706
50%      260.000000
75%      393.000000
max      790.000000

In [9]:
# Validate data
print("Checking for missing values in training data:")
print(X.isnull().sum())
print("\nChecking for missing values in new compositions:")
print(X_new.isnull().sum())

print("\nValue ranges in training data:")
print(X.describe())
print("\nValue ranges in new compositions:")
print(X_new.describe())

Checking for missing values in training data:
Pb                        0
Ni                        0
Si                        0
Sn                        0
V                         0
Mn                        0
Cr                        0
aging_time                0
solution_temp             0
Zr                        0
Cu                        0
strain_hardening_index    0
Mg                        0
Ti                        0
aging_temp                0
Al                        0
Fe                        0
solution_time             0
Zn                        0
Co                        0
dtype: int64

Checking for missing values in new compositions:
Pb                        0
Ni                        0
Si                        0
Sn                        0
V                         0
Mn                        0
Cr                        0
aging_time                0
solution_temp             0
Zr                        0
Cu                        0
strain_hardening_index 

In [10]:
# Function to create and optimize a multi-output model for all properties
def train_and_optimize_model(X, y_dict):
    print("\nTraining multi-output model for all properties")
    
    # Combine all target variables into a single array
    y_combined = np.column_stack([y_dict[target] for target in target_variables])
    
    # Initialize best score and model
    best_score = float('-inf')
    best_model = None
    best_model_name = None
    
    # Create cross-validation object
    kf = KFold(n_splits=5, shuffle=True, random_state=42)
    
    # Define objective functions for each model type
    def objective_xgb(trial):
        params = {
            'n_estimators': trial.suggest_int('n_estimators', 50, 500),
            'max_depth': trial.suggest_int('max_depth', 3, 8),
            'learning_rate': trial.suggest_float('learning_rate', 0.01, 0.1, log=True),
            'min_child_weight': trial.suggest_int('min_child_weight', 1, 5),
            'subsample': trial.suggest_float('subsample', 0.6, 1.0),
            'colsample_bytree': trial.suggest_float('colsample_bytree', 0.6, 1.0)
        }
        
        # Use MultiOutputRegressor for XGBoost
        model = MultiOutputRegressor(xgb.XGBRegressor(**params, random_state=42))
        scores = cross_val_score(model, X, y_combined, cv=kf, scoring='r2')
        return scores.mean()

    def objective_lgb(trial):
        params = {
            'n_estimators': trial.suggest_int('n_estimators', 50, 500),
            'max_depth': trial.suggest_int('max_depth', 3, 8),
            'learning_rate': trial.suggest_float('learning_rate', 0.01, 0.1, log=True),
            'num_leaves': trial.suggest_int('num_leaves', 10, 50),
            'subsample': trial.suggest_float('subsample', 0.6, 1.0),
            'colsample_bytree': trial.suggest_float('colsample_bytree', 0.6, 1.0)
        }
        
        # Use MultiOutputRegressor for LightGBM
        model = MultiOutputRegressor(lgb.LGBMRegressor(**params, random_state=42))
        scores = cross_val_score(model, X, y_combined, cv=kf, scoring='r2')
        return scores.mean()

    # Optimize each model type with fewer trials
    for model_name, objective_func in [
        ('XGBoost', objective_xgb),
        ('LightGBM', objective_lgb)
    ]:
        print(f"\nOptimizing {model_name}...")
        study = optuna.create_study(direction='maximize')
        study.optimize(objective_func, n_trials=10, show_progress_bar=True)  # Reduced from 20 to 10 trials
        
        if study.best_value > best_score:
            best_score = study.best_value
            best_model_name = model_name
            best_params = study.best_params
            
            # Train the final model with best parameters
            if model_name == 'XGBoost':
                best_model = MultiOutputRegressor(xgb.XGBRegressor(**best_params, random_state=42))
            else:  # LightGBM
                best_model = MultiOutputRegressor(lgb.LGBMRegressor(**best_params, random_state=42))
    
    # Train the best model on full dataset
    print(f"\nBest model: {best_model_name} with average R² score: {best_score:.4f}")
    best_model.fit(X, y_combined)
    return best_model, target_variables

# Dictionary to store models
models = {}

In [11]:
# Train and optimize a single multi-output model for all properties
print("\nTraining multi-output model...")
model, target_list = train_and_optimize_model(X, y_dict)
print("Finished training model")

# Make predictions for new compositions
predictions_array = model.predict(X_new)
predictions = {target: predictions_array[:, i] for i, target in enumerate(target_list)}

[I 2025-10-09 02:53:21,104] A new study created in memory with name: no-name-2f5be0ab-3dac-4e09-b244-670a8bb23fec



Training multi-output model...

Training multi-output model for all properties

Optimizing XGBoost...


Best trial: 0. Best value: 0.56556:  10%|█         | 1/10 [00:06<01:02,  6.91s/it]

[I 2025-10-09 02:53:28,000] Trial 0 finished with value: 0.565560361569872 and parameters: {'n_estimators': 429, 'max_depth': 3, 'learning_rate': 0.02105359307505453, 'min_child_weight': 2, 'subsample': 0.7788925579897241, 'colsample_bytree': 0.6834998832554479}. Best is trial 0 with value: 0.565560361569872.


Best trial: 0. Best value: 0.56556:  20%|██        | 2/10 [00:19<01:21, 10.14s/it]

...

Best trial: 5. Best value: 0.603906:  90%|█████████ | 9/10 [00:26<00:02,  2.81s/it]

[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No f

Best trial: 5. Best value: 0.603906: 100%|██████████| 10/10 [00:34<00:00,  3.43s/it]

[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No f

[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No f

In [18]:
# Predict all properties using the best multi-output model
y_pred = model.predict(X_new)

# Create a DataFrame with predictions
predictions_df = pd.DataFrame(y_pred, columns=target_variables)

# Get the minimum values from the training data for each property
min_elongation = final_data['Elongation (%)'].min()
min_tensile = final_data['Tensile Strength (MPa)'].min()
min_yield = final_data['Yield Strength (MPa)'].min()

# Find improved compositions
improved_compositions = (
    (predictions_df['Elongation (%)'] > min_elongation) &
    (predictions_df['Tensile Strength (MPa)'] > min_tensile) &
    (predictions_df['Yield Strength (MPa)'] > min_yield)
)

# Combine predictions with input compositions
results = pd.concat([new_compositions.reset_index(drop=True), predictions_df], axis=1)

# Show and save improved results
print("\nNumber of compositions with improved properties:", improved_compositions.sum())
print("\nImproved compositions:")
print(results[improved_compositions])

# Save the full results
results.to_csv('predicted_properties.csv', index=False)
print("\nResults saved to 'predicted_properties.csv'")



Number of compositions with improved properties: 100

Improved compositions:
           Al        Cu        Mg        Mn        Si        Zn        Fe  \
0   92.472440  0.863421  2.768146  0.091957  0.000000  3.412003  0.071136   
1   95.589066  0.423996  1.794946  0.124174  1.367767  0.506073  0.116012   
2   97.034935  0.000000  0.265492  0.000000  2.240273  0.045864  0.097513   
3   91.274670  1.025460  2.503181  0.093533  0.531582  4.285638  0.089350   
4   95.635020  2.024474  0.976213  0.329061  0.496437  0.184357  0.196607   
..        ...       ...       ...       ...       ...       ...       ...   
95  97.907970  0.328434  1.156293  0.149798  0.142787  0.127408  0.074561   
96  85.529690  0.656599  2.449903  0.114455  5.957502  5.008502  0.108286   
97  97.782470  0.291934  1.215404  0.144213  0.147062  0.161154  0.072275   
98  97.406975  0.537926  0.205165  0.798736  0.783493  0.000000  0.215800   
99  97.029590  0.168405  1.333647  1.011887  0.000000  0.000000  0.276627  